In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# LLM Classification Finetuning — Kaggle Notebook (fast numeric baseline, convergence-safe)
# Runs end-to-end on Kaggle and always creates /kaggle/working/submission.csv

import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
# from sklearn.preprocessing import StandardScaler  # enable if you want to standardize

# 1) Kaggle input paths
TRAIN_PATH  = "/kaggle/input/llm-classification-finetuning/train.csv"
TEST_PATH   = "/kaggle/input/llm-classification-finetuning/test.csv"
SAMPLE_PATH = "/kaggle/input/llm-classification-finetuning/sample_submission.csv"
OUT_PATH    = "/kaggle/working/submission.csv"

print("Checking inputs:")
print(" train :", TRAIN_PATH,  "->", os.path.exists(TRAIN_PATH))
print(" test  :", TEST_PATH,   "->", os.path.exists(TEST_PATH))
print(" sample:", SAMPLE_PATH, "->", os.path.exists(SAMPLE_PATH))

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError("test.csv not found in Kaggle inputs. Click '+ Add data' and attach the competition dataset.")

df_test = pd.read_csv(TEST_PATH)
has_train = os.path.exists(TRAIN_PATH)
df_train = pd.read_csv(TRAIN_PATH) if has_train else None

print("Test shape:", df_test.shape)
if df_train is not None:
    print("Train shape:", df_train.shape)

# 2) Lightweight numeric features
def tx(s):
    return s.fillna("").astype(str)

def build_numeric_features(df):
    p = tx(df["prompt"])
    a = tx(df["response_a"])
    b = tx(df["response_b"])

    # lengths
    p_len = p.str.len().to_numpy().reshape(-1, 1)
    a_len = a.str.len().to_numpy().reshape(-1, 1)
    b_len = b.str.len().to_numpy().reshape(-1, 1)

    # word counts
    p_wc = p.str.split().apply(len).to_numpy().reshape(-1, 1)
    a_wc = a.str.split().apply(len).to_numpy().reshape(-1, 1)
    b_wc = b.str.split().apply(len).to_numpy().reshape(-1, 1)

    # differences and ratios (A vs B)
    len_diff = a_len - b_len
    wc_diff  = a_wc - b_wc
    len_ratio = (a_len + 1) / (b_len + 1)
    wc_ratio  = (a_wc + 1) / (b_wc + 1)

    X = np.hstack([
        p_len, a_len, b_len,
        p_wc,  a_wc,  b_wc,
        len_diff, wc_diff,
        len_ratio, wc_ratio
    ]).astype("float32")
    return X

X_test = build_numeric_features(df_test)

# 3) Train if targets exist; else use uniform probabilities
use_scaler = False  # set to True to standardize features (may help, slightly slower)

if df_train is not None and all(c in df_train.columns for c in ["winner_model_a","winner_model_b","winner_tie"]):
    X_train = build_numeric_features(df_train)
    y = df_train[["winner_model_a","winner_model_b","winner_tie"]].to_numpy().argmax(axis=1)

    # Optional standardization
    if use_scaler:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler(with_mean=True)  # numeric dense features -> with_mean=True is fine
        X_train_std = scaler.fit_transform(X_train)
        X_test_std  = scaler.transform(X_test)
        X_tr_use, X_te_use = X_train_std, X_test_std
    else:
        X_tr_use, X_te_use = X_train, X_test

    print("Training multinomial LogisticRegression (numeric-only, max_iter=1000)...")
    clf = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=1000,     # increased to avoid convergence warnings
        n_jobs=-1
    )
    clf.fit(X_tr_use, y)
    preds = clf.predict_proba(X_te_use)
else:
    print("No train targets available — using uniform probabilities.")
    preds = np.ones((len(df_test), 3), dtype="float32") / 3.0

# 4) Build and save submission
sub = pd.DataFrame({
    "id": df_test["id"],
    "winner_model_a": preds[:, 0],
    "winner_model_b": preds[:, 1],
    "winner_tie": preds[:, 2],
})

# clip + renormalize to ensure valid probabilities
probs = sub[["winner_model_a","winner_model_b","winner_tie"]].to_numpy()
probs = np.clip(probs, 1e-9, 1.0)
probs = probs / probs.sum(axis=1, keepdims=True)
sub[["winner_model_a","winner_model_b","winner_tie"]] = probs

sub.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
print(sub.head())

# 5) List working directory to help Kaggle UI show the file
print("\nFiles in /kaggle/working:")
for f in os.listdir("/kaggle/working"):
    print("-", f)

# LLM Preference Prediction — Kaggle Fast Baseline

## Overview
This project provides a **fast, numeric-only baseline** for the Kaggle competition *“LLM Classification Finetuning.”*  
The goal is to predict user preference between two chatbot responses (A, B, or Tie).  

I built this as a **quick starting point** to test how far simple numeric heuristics (like lengths and word counts) can go before moving on to embeddings or transformer-based models.  
The provided Python script is designed for **Kaggle Notebooks** and ensures a valid `submission.csv` is always generated.

---

## Key Features
- **Kaggle-Ready:** Uses correct input paths (`/kaggle/input/...`) for seamless execution.
- **Ultra-Fast:** Employs only lightweight numeric features (text lengths, word counts, and their ratios/differences).
- **Robust:**
  - Trains a **Logistic Regression** model if `train.csv` is available.
  - Falls back to **uniform probabilities (1/3, 1/3, 1/3)** if `train.csv` is not found or lacks target columns.
- **Convergence Fixed:** `max_iter` for Logistic Regression is increased to prevent convergence warnings.
- **Guaranteed Output:** Always creates `submission.csv` in `/kaggle/working`.

---

## How to Use
1. **Create a New Kaggle Notebook:** Go to the competition page → `Code` → `New Notebook`.
2. **Attach Data:** Ensure the competition dataset (*LLM Classification Finetuning*) is attached (check the `Input` section on the right).
3. **Paste Code:** Copy the entire provided Python script into a code cell in your Kaggle Notebook.
4. **Run:** Click **Run All** (or execute the cells sequentially).
5. **Submit:**
   - Click `Save Version` → `Save & Run All`.
   - Once the run completes, the **Submit to Competition** button will be enabled on the version page.

---

## Model Details
- **Features:** Character lengths, word counts, and derived ratios/differences for `prompt`, `response_a`, and `response_b`.
- **Model:** Multinomial Logistic Regression (`solver='lbfgs'`, `max_iter=1000`).
- **Output:** `submission.csv` with `id`, `winner_model_a`, `winner_model_b`, `winner_tie` columns.  
  Probabilities are clipped and renormalized to ensure stability.

Typical public leaderboard score: around **0.69–0.71 log loss** (as a quick sanity baseline).

---

## Troubleshooting
- **`FileNotFoundError`:** Ensure the competition dataset is correctly attached to your Kaggle Notebook.
- **Convergence Warning:** This is now addressed by increasing `max_iter`. If it persists, it's usually harmless for this baseline.
- **No `submission.csv`:** Verify the script ran to completion and check the `/kaggle/working` directory (the script prints its contents at the end).

---

## Final Notes
That’s it — a clean, fast, and fully reproducible baseline to get you started!  
Feel free to extend it with richer features or embeddings.

**Author:** Heittor Lopes Valadares  
**Competition:** [LLM Classification Finetuning]()
